# SAMOS Run8 Dolidze25 — Step08 to End Notebook

This notebook runs the current pipeline from **Step08 onward** for **Run8 Dolidze25**, with QC after each major stage and key diagnostics displayed inline.

Current intended order:
- **08a** ridge-guided extraction for EVEN and ODD
- **08b** merge EVEN and ODD
- **08c** attach wavelength
- **09a / 09b / 09bm / 09c** OH refinement and optional cleanup
- **10a / 10b** telluric template and application
- **11a / 11b / 11c** metadata, SkyMapper, flux calibration

Design choices:
- run from the **repo root**
- display key output files and PNG diagnostics inline
- **skip missing optional scripts cleanly**
- let real stage failures surface so you can assess the current scripts

In [1]:
# Imports and Definitions

from pathlib import Path
import os
import sys
import subprocess
import shlex
import glob
from IPython.display import display, Markdown, Image

def find_repo_root(start: Path):
    candidates = [start] + list(start.parents)
    for p in candidates:
        cfg_dir = p / "config"
        if cfg_dir.is_dir() and (cfg_dir / "__init__.py").exists():
            return p
    return None

repo_env = os.environ.get("SAMOS_REPO_ROOT", "").strip()
if repo_env:
    REPO_ROOT = Path(repo_env).expanduser().resolve()
else:
    REPO_ROOT = find_repo_root(Path.cwd())

if REPO_ROOT is None:
    raise RuntimeError(
        "Could not find the SAMOS repository root. "
        "Run this notebook from inside the repo tree, or set SAMOS_REPO_ROOT."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import config

display(Markdown(f'''
**Repo root:** `{REPO_ROOT}`  
**ST06_SCIENCE:** `{config.ST06_SCIENCE}`  
**ST07_WAVECAL:** `{config.ST07_WAVECAL}`  
**ST08_EXTRACT1D:** `{config.ST08_EXTRACT1D}`  
'''))


**Repo root:** `/Users/robberto/Library/CloudStorage/Box-Box/My Documents - Massimo Robberto/@Massimo/_Science/2. Projects_HW/2017.SAMOS/samos-pipeline`  
**ST06_SCIENCE:** `/Users/robberto/Library/CloudStorage/Box-Box/My Documents - Massimo Robberto/@Massimo/_Science/2. Projects_HW/2017.SAMOS/_Run8_Science_2026_01/SAMI/Dolidze25/reduced/06_science`  
**ST07_WAVECAL:** `/Users/robberto/Library/CloudStorage/Box-Box/My Documents - Massimo Robberto/@Massimo/_Science/2. Projects_HW/2017.SAMOS/_Run8_Science_2026_01/SAMI/Dolidze25/reduced/07_wavecal`  
**ST08_EXTRACT1D:** `/Users/robberto/Library/CloudStorage/Box-Box/My Documents - Massimo Robberto/@Massimo/_Science/2. Projects_HW/2017.SAMOS/_Run8_Science_2026_01/SAMI/Dolidze25/reduced/08_extract1d`  


In [2]:
# Helpers

def _env():
    env = os.environ.copy()
    env["PYTHONPATH"] = str(REPO)
    return env

def run_py(script, args="", required=True, title=None):
    script = Path(script)
    if not script.is_absolute():
        script = REPO_ROOT / script

    msg = title or script.name
    print(f"\n=== {msg} ===")

    if not script.exists():
        text = f"[SKIP] Missing script: {script}"
        if required:
            raise FileNotFoundError(text)
        print(text)
        return None

    cmd = [sys.executable, str(script)]
    if args:
        cmd += shlex.split(args)

    print("[CMD]", " ".join(shlex.quote(c) for c in cmd))

    res = subprocess.run(
        cmd,
        cwd=str(REPO_ROOT),
        env={**os.environ, "PYTHONPATH": str(REPO_ROOT)},
        text=True,
        capture_output=True,
    )

    print("\n--- STDOUT ---")
    print(res.stdout if res.stdout else "[empty]")

    print("\n--- STDERR ---")
    print(res.stderr if res.stderr else "[empty]")

    if res.returncode != 0:
        raise RuntimeError(f"{script.name} failed with exit code {res.returncode}")

    return res

def run_py_optional(script, args="", title=None):
    return run_py(script, args=args, required=False, title=title)

def latest(pattern):
    hits = sorted(glob.glob(str(pattern)))
    return Path(hits[-1]) if hits else None

def show_images(pattern, maxn=6, width=900):
    hits = sorted(glob.glob(str(pattern)))
    if not hits:
        print(f"[no images] {pattern}")
        return
    for p in hits[:maxn]:
        print(p)
        display(Image(filename=p, width=width))

def show_last_images(pattern, maxn=6, width=900):
    hits = sorted(glob.glob(str(pattern)))
    if not hits:
        print(f"[no images] {pattern}")
        return
    for p in hits[-maxn:]:
        print(p)
        display(Image(filename=p, width=width))

# TO DISPLAY A RANGE OF FILES
import re
from pathlib import Path
from IPython.display import display, Image
import glob

def _slit_key(path):
    name = Path(path).name
    m = re.search(r"SLIT(\d+)", name.upper())
    if m:
        return (0, int(m.group(1)), name)
    return (1, 10**9, name)


def show_slit_range(pattern, slit_min=None, slit_max=None, maxn=None, width=900):
    hits = glob.glob(str(pattern))
    if not hits:
        print(f"[no images] {pattern}")
        return

    hits = sorted(hits, key=_slit_key)

    selected = []
    for p in hits:
        m = re.search(r"SLIT(\d+)", Path(p).name.upper())
        if m:
            slit = int(m.group(1))
            if slit_min is not None and slit < slit_min:
                continue
            if slit_max is not None and slit > slit_max:
                continue
        selected.append(p)

    if maxn is not None:
        selected = selected[:maxn]

    if not selected:
        print("[no images in selected range]")
        return

    for p in selected:
        print(p)
        display(Image(filename=p, width=width))

        

def print_exists(label, p):
    p = Path(p)
    print(f"{label}: {p}  {'[OK]' if p.exists() else '[MISSING]'}")

# Downstream run: current pipeline after Step08

This section runs the current scripts if they exist in the repo. Optional stages are skipped cleanly when absent.

In [ ]:

downstream = [
    ("09a",  "pipeline/step09_oh_refine/step09a_measure_oh_shifts.py", "", True),
    ("09b",  "pipeline/step09_oh_refine/step09b_apply_oh_shifts.py", "", True),
    ("09bm", "pipeline/step09_oh_refine/step09b_apply_oh_shifts.py", "--manual", False),
    ("09c",  "pipeline/step09_oh_refine/step09_abab_driver.py", "", False),
    ("10a",  "pipeline/step10_telluric/step10a_build_telluric_template.py", "", True),
    ("10b",  "pipeline/step10_telluric/step10b_apply_telluric.py", "", True),
    ("11a",  "pipeline/step11_fluxcal/step11a_extract_header_radec_resilient.py", "", True),
    ("11b",  "pipeline/step11_fluxcal/step11b_query_skymapper.py", "", True),
    ("11c",  "pipeline/step11_fluxcal/step11c_fluxcal.py", "", True),
]

for key, script, args, required in downstream:
    if required:
        run_py(script, args=args, title=key)
    else:
        run_py_optional(script, args=args, title=f"{key} (optional)")

## QC after downstream stages

Current QC associations:
- **09b** → `qc/step10/qc_step10_oh.py`
- **10b** → `qc/step09/qc_step09_telluric.py`
- **11c** → `qc/step11/qc_step11_grid.py`, `qc/step11/qc_step11_summary.py`

In [ ]:

run_py_optional("qc/step10/qc_step10_oh.py", title="QC 09b / OH refine")
run_py_optional("qc/step09/qc_step09_telluric.py", title="QC 10b / telluric")
run_py_optional("qc/step11/qc_step11_grid.py", title="QC 11c grid")
run_py_optional("qc/step11/qc_step11_summary.py", title="QC 11c summary")

In [ ]:

candidate_dirs = []
for attr in ["ST09_OH_REFINE", "ST09_TELLURIC", "ST10_TELLURIC", "ST10_OH_REFINE", "ST11_FLUXCAL"]:
    if hasattr(config, attr):
        candidate_dirs.append(Path(getattr(config, attr)))

for d in candidate_dirs:
    if d.exists():
        print(f"\nDiagnostics under: {d}")
        show_last_images(d / "*.png", maxn=6, width=900)

## Final product checks

This cell lists the expected main outputs from Step08 to Step11 and whether they currently exist.

In [ ]:

checks = [
    ("08a EVEN", st08 / "extract1d_optimal_ridge_even.fits"),
    ("08a ODD",  st08 / "extract1d_optimal_ridge_odd.fits"),
    ("08b ALL",  st08 / "extract1d_optimal_ridge_all.fits"),
    ("08c WAV",  st08 / "extract1d_optimal_ridge_all_wav.fits"),
]

for name, path in checks:
    print_exists(name, path)

for attr in ["ST09_OH_REFINE", "ST09_TELLURIC", "ST10_TELLURIC", "ST10_OH_REFINE", "ST11_FLUXCAL"]:
    if hasattr(config, attr):
        d = Path(getattr(config, attr))
        print(f"\n{attr}: {d}")
        if d.exists():
            for p in sorted(d.glob("*.fits"))[:20]:
                print("  ", p.name)

## Notes

- Run this notebook from the **samos-pipeline repo root**.
- It uses the **canonical pipeline scripts** under `pipeline/...` and `qc/...`.
- A next pass can make it more compact and presentation-style, keeping only the most informative figures.